# Rental Prices in Bahrain’s Real Estate Market
### Project 3 - Data Science BootCamp Competition


**Problem Statement:** the Kingdom’s real estate market continues to evolve with the 2030 economic vision, understanding property value has become a data-driven science.

**Notebook roadmap:**
1. Data Collection - load and take a first look at all raw data sources
2. Data Cleaning - handle missing data, fix types, engineer features, merge datasets
3. Train & testing
4. Modeling  



# Imports

In [55]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV, ElasticNet
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# Load the Data

In [21]:
train_raw = pd.read_csv('data.csv')
test_raw = pd.read_csv('test.csv')

print('train:', train_raw.shape)
print('test:', test_raw.shape)
train_raw.head()

train: (10578, 16)
test: (3527, 15)


,Property_id,Offer,URL,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent
0,6046,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3+ Maid,4,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0
1,2240,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0
2,6248,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,studio,1,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0
3,7177,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0
4,7842,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5+ Maid,5,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0


### Data Dictionary

Features include:

Feature	Description
Property_id	A unique identifier for each listing.
Offer	The type of listing (e.g., For Rent).
URL	The original source link for the listing.
Property_type	Type of residence (e.g., Apartment, Villa, Studio).
Include_w_e	Whether Electricity and Water (EWA) are included in the price.
Title	The headline of the listing (contains descriptive keywords).
Area	The specific neighborhood (e.g., Juffair, Saar, Amwaj).
Governorate	The administrative region (e.g., Capital, Muharraq, Northern, Southern).
Beds	Number of bedrooms.
Baths	Number of bathrooms.
Size	The total area of the property.
Availability_date	When the property is ready for move-in.
Agent_name	The name of the listing agent.
Agency	The name of the agency.
Amenities	The number of features (e.g., Swimming Pool, Gym -> 2).
rent	Target Variable. The monthly rental price in BHD.

In [22]:
train_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 10578 entries, 0 to 10577
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Property_id        10578 non-null  int64  
 1   Offer              10578 non-null  str    
 2   URL                10578 non-null  str    
 3   Property_type      10578 non-null  str    
 4   Include_w_e        10578 non-null  str    
 5   Title              10575 non-null  str    
 6   Area               10575 non-null  str    
 7   Governorate        10575 non-null  str    
 8   Beds               10578 non-null  str    
 9   Baths              10578 non-null  str    
 10  Size               10578 non-null  str    
 11  Availability_date  10059 non-null  str    
 12  Agent_name         10575 non-null  str    
 13  Agency             10575 non-null  str    
 14  Amenities          10272 non-null  float64
 15  rent               10575 non-null  float64
dtypes: float64(2), int64(1), str(13)


In [24]:
train_raw.isnull().sum()

Property_id            0
Offer                  0
URL                    0
Property_type          0
Include_w_e            0
Title                  3
Area                   3
Governorate            3
Beds                   0
Baths                  0
Size                   0
Availability_date    519
Agent_name             3
Agency                 3
Amenities            306
rent                   3
dtype: int64

In [25]:
train_raw.dtypes

Property_id            int64
Offer                    str
URL                      str
Property_type            str
Include_w_e              str
Title                    str
Area                     str
Governorate              str
Beds                     str
Baths                    str
Size                     str
Availability_date        str
Agent_name               str
Agency                   str
Amenities            float64
rent                 float64
dtype: object

In [26]:
train_raw.shape

(10578, 16)

# 2. EDA 


Checking the distribution before doing anything else, since this is what everything downstream depends on

### 1 dropping rows

In [29]:
broken = train_raw[train_raw['rent'].isnull()]
print(broken)
train_raw = train_raw.dropna(subset=['rent']).reset_index(drop=True)
print('train shape after dropping broken rows:', train_raw.shape)

      Property_id Offer                                                URL  \
5194          125  Rent  https://www.propertyfinder.bh/en/plp/rent/apar...   
6787           25  Rent  https://www.propertyfinder.bh/en/plp/rent/apar...   
7864         9012  Rent  https://www.propertyfinder.bh/en/plp/rent/apar...   

     Property_type Include_w_e Title Area Governorate Beds Baths  \
5194     Apartment   Exclusive   NaN  NaN         NaN    2     4   
6787     Apartment   Exclusive   NaN  NaN         NaN    2     4   
7864     Apartment   Inclusive   NaN  NaN         NaN    2     2   

                      Size Availability_date Agent_name Agency  Amenities  \
5194  1,399 sqft / 130 sqm               NaN        NaN    NaN        5.0   
6787  1,399 sqft / 130 sqm               NaN        NaN    NaN        5.0   
7864  1,184 sqft / 110 sqm               NaN        NaN    NaN        8.0   

      rent  
5194   NaN  
6787   NaN  
7864   NaN  
train shape after dropping broken rows: (10575, 16)


### 2Categorical fields

In [32]:
for c in ['Property_type', 'Include_w_e', 'Governorate', 'Beds', 'Baths']:
    print(c,sorted(train_raw[c].dropna().unique().tolist()))
    print()

Property_type ['Apartment', 'Bulk Rent Unit', 'Bungalow', 'Compound', 'Duplex', 'Hotel Apartment', 'Penthouse', 'Short Term & Hotel Apartment', 'Townhouse', 'Villa', 'Whole Building']

Include_w_e ['Exclusive', 'Inclusive']

Governorate ['Capital Governorate', 'Central Governorate', 'Muharraq Governorate', 'Northern Governorate', 'Southern Governorate']

Beds ['0', '1', '1+ Maid', '2', '2+ Maid', '3', '3+ Maid', '4', '4+ Maid', '5', '5+ Maid', '6', '6+ Maid', '7', '7+', '7+ Maid', '7++ Maid', 'studio', 'studio+ Maid']

Baths ['1', '2', '3', '4', '5', '6', '7', '7+', 'none']



`Beds` and `Baths` are stored as messy strings (`'3+ Maid'`, `'studio'`, `'7+'`, `'none'`) instead of numbers 

In [33]:
print(train_raw['Size'].head())
print()
size_train = (train_raw['Size'] == '11 sqft / 1 sqm').sum()
size_test = (test_raw['Size'] == '11 sqft / 1 sqm').sum()
print(f"'11 sqft / 1 sqm' rows train: {size_train}, test: {size_test}")

0    2,368 sqft / 220 sqm
1    3,229 sqft / 300 sqm
2       484 sqft / 45 sqm
3    1,507 sqft / 140 sqm
4    4,844 sqft / 450 sqm
Name: Size, dtype: str

'11 sqft / 1 sqm' rows train: 112, test: 34


In [34]:
def has_maid(x):
    # any string mentioning "Maid" gets flagged
    return 0 if pd.isna(x) else int('Maid' in x)


def bed_number(x):
    if pd.isna(x):
        return np.nan
    x = x.replace(' Maid', '').replace('+', '')  # strips maid tag and any '+'
    if x == 'studio':
        return 0
    if x == '7':  # covers what used to be '7+' and '7++' before stripping '+'
        return 7
    return float(x)


def parse_baths(x):
    if pd.isna(x):
        return np.nan
    if x == 'none':
        return 0
    return 7.0 if x == '7+' else float(x)


def parse_size(x):
    # '2,368 sqft / 220 sqm' -> 220.0. '11 sqft / 1 sqm' is a placeholder for missing size.
    if pd.isna(x) or x == '11 sqft / 1 sqm':
        return np.nan
    sqm_part = x.split('/')[1]
    return float(sqm_part.replace('sqm', '').replace(',', '').strip())


def clean_base(df):
    df = df.copy()

    df['has_maid_room'] = df['Beds'].apply(has_maid)
    df['is_studio'] = df['Beds'].str.startswith('studio').astype(int)
    df['bed_count'] = df['Beds'].apply(bed_number)
    df['bath_count'] = df['Baths'].apply(parse_baths)
    df['sqm'] = df['Size'].apply(parse_size)

    avail = pd.to_datetime(df['Availability_date'], format='%d %b %Y', errors='coerce')
    df['avail_month'] = avail.dt.month.fillna(0).astype(int)
    df['avail_missing'] = avail.isna().astype(int)

    # 'Sub-area, Broad area' -> keep the broad area, it's low-cardinality enough to one-hot
    df['broad_area'] = df['Area'].str.split(',').str[-1].str.strip()

    df['title_len'] = df['Title'].str.len()
    df['title_word_count'] = df['Title'].str.split().str.len()
    df['Amenities'] = df['Amenities'].fillna(0)
    return df


train = clean_base(train_raw)
test = clean_base(test_raw)

print(train[['Beds', 'bed_count', 'has_maid_room', 'is_studio']].drop_duplicates().sort_values('bed_count'))

              Beds  bed_count  has_maid_room  is_studio
2           studio        0.0              0          1
848   studio+ Maid        0.0              1          1
1214             0        0.0              0          0
9                1        1.0              0          0
229        1+ Maid        1.0              1          0
10         2+ Maid        2.0              1          0
5                2        2.0              0          0
0          3+ Maid        3.0              1          0
3                3        3.0              0          0
23               4        4.0              0          0
6          4+ Maid        4.0              1          0
1                5        5.0              0          0
4          5+ Maid        5.0              1          0
112              6        6.0              0          0
479        6+ Maid        6.0              1          0
364             7+        7.0              0          0
1467      7++ Maid        7.0              1    

In [35]:
sqm_medians = train.groupby('Property_type')['sqm'].median()
train['sqm'] = train.apply(lambda r: sqm_medians[r['Property_type']] if pd.isna(r['sqm']) else r['sqm'], axis=1)
test['sqm'] = test.apply(
    lambda r: sqm_medians.get(r['Property_type'], sqm_medians.median()) if pd.isna(r['sqm']) else r['sqm'], axis=1
)
print('remaining sqm NaNs -> train:', train['sqm'].isna().sum(), ' test:', test['sqm'].isna().sum())

remaining sqm NaNs -> train: 0  test: 0


## 5. Outlier removal 

In [36]:
train['rent_per_sqm'] = train['rent'] / train['sqm']
print(train['rent_per_sqm'].describe(percentiles=[.5, .9, .95, .99, .999]))

count    10575.000000
mean         4.601784
std         21.548427
min          0.308511
50%          3.500000
90%          6.250000
95%          7.236842
99%          9.814815
99.9%      348.455096
max       1045.454545
Name: rent_per_sqm, dtype: float64


In [37]:
outliers = train[train['rent_per_sqm'] > 15].sort_values('rent_per_sqm', ascending=False)
print(f'{len(outliers)} rows above the 15 BHD/sqm/month cutoff out of {len(train)} total')
outliers[['Property_id', 'Property_type', 'sqm', 'rent', 'rent_per_sqm']]

23 rows above the 15 BHD/sqm/month cutoff out of 10575 total


,Property_id,Property_type,sqm,rent,rent_per_sqm
7275,12923,Apartment,66.0,69000.0,1045.454545
292,657,Apartment,94.0,72000.0,765.957447
7193,10705,Villa,545.0,400000.0,733.944954
8028,11669,Apartment,43.0,31500.0,732.558140
8945,7480,Villa,2.0,1350.0,675.000000
9015,11623,Apartment,133.0,86000.0,646.616541
9396,13289,Villa,375.0,230000.0,613.333333
4397,11498,Villa,273.0,130000.0,476.190476
3810,10706,Villa,120.0,55000.0,458.333333
8523,13785,Apartment,91.0,36400.0,400.000000


## 6. Final feature set

In [41]:
area_freq = train['Area'].value_counts()  # count how many times each Area shows up in train
train['area_freq'] = train['Area'].map(area_freq).fillna(0)  # look up that count for every train row
test['area_freq'] = test['Area'].map(area_freq).fillna(0)  # same lookup for test, using train's counts

agency_freq = train['Agency'].value_counts()  # count how many times each Agency shows up in train
train['agency_freq'] = train['Agency'].map(agency_freq).fillna(0)  # look up that count for every train row
test['agency_freq'] = test['Agency'].map(agency_freq).fillna(0)  # same lookup for test, using train's counts

# list of all numeric columns to feed into the model
num_features = ['bed_count', 'bath_count', 'sqm', 'Amenities', 'avail_month',
                 'has_maid_room', 'is_studio', 'avail_missing',
                 'title_len', 'title_word_count', 'area_freq', 'agency_freq']

skewed_features = ['sqm', 'Amenities']  # these two are right-skewed, need a PowerTransformer later

cat_features = ['Property_type', 'Include_w_e', 'Governorate', 'broad_area']  # columns to one-hot encode

print('numeric features:', num_features)  # just printing the final lists to eyeball them
print('categorical features:', cat_features)

numeric features: ['bed_count', 'bath_count', 'sqm', 'Amenities', 'avail_month', 'has_maid_room', 'is_studio', 'avail_missing', 'title_len', 'title_word_count', 'area_freq', 'agency_freq']
categorical features: ['Property_type', 'Include_w_e', 'Governorate', 'broad_area']


In [42]:
X = train[num_features + cat_features]
y = np.log1p(train['rent']) 
X_test_final = test[num_features + cat_features]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape, X_test_final.shape)

(8441, 16) (2111, 16) (3527, 16)


# preprocessing

In [43]:
plain_num = [c for c in num_features if c not in skewed_features]

preprocess = ColumnTransformer([
    ('skewed', Pipeline([('power', PowerTransformer()), ('scale', StandardScaler())]), skewed_features),
    ('plain', StandardScaler(), plain_num),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
])

# model comparison

In [45]:
reg_results = []  

def evaluate(name, model, best_alpha=None):
    pipe = Pipeline([('prep', preprocess), ('model', model)])  
    pipe.fit(X_train, y_train)  # train on the training split, target is log1p(rent)
    pred = np.expm1(pipe.predict(X_val))  # predict on the held-out val set, undo the log to get back to BHD
    actual = np.expm1(y_val)  # undo the log on the true values too, so we compare on the same scale
    rmse = mean_squared_error(actual, pred) ** 0.5  
    mae = mean_absolute_error(actual, pred)  # mean absolute error, in BHD 
    r2 = r2_score(actual, pred)  # how much of the variance in rent this model explains
    reg_results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2, 'Best Alpha': best_alpha})  
    print(f'{name:22s} RMSE={rmse:8.2f}  MAE={mae:8.2f}  R2={r2:.4f}')  
    return pipe  

In [53]:
evaluate('Linear Regression', LinearRegression())

ridge = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
p = evaluate('Ridge', ridge)
evaluate.__globals__['reg_results'][-1]['Best Alpha'] = p.named_steps['model'].alpha_

lasso = LassoCV(alphas=np.logspace(-3, 3, 100), cv=5, max_iter=20000)
p = evaluate('Lasso', lasso)
reg_results[-1]['Best Alpha'] = p.named_steps['model'].alpha_

enet = ElasticNetCV(alphas=np.logspace(-3, 3, 100), cv=5, max_iter=20000)
p = evaluate('ElasticNet', enet)
reg_results[-1]['Best Alpha'] = p.named_steps['model'].alpha_

Linear Regression      RMSE=  215.98  MAE=  121.53  R2=0.6697
Ridge                  RMSE=  211.78  MAE=  121.13  R2=0.6824
Lasso                  RMSE=  206.24  MAE=  122.12  R2=0.6988
ElasticNet             RMSE=  204.64  MAE=  120.95  R2=0.7035


In [58]:
final_model = Pipeline([('prep', preprocess), ('model', ElasticNetCV(
    alphas=np.logspace(-3, 3, 100), cv=5, max_iter=20000, random_state=42
))])
final_model.fit(X, y)  # fit on all of train, not just X_train

test_pred_log = final_model.predict(X_test_final)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({'Property_id': test['Property_id'], 'rent': test_pred})
print(submission.shape)
submission.head()

(3527, 2)


,Property_id,rent
0,4394,416.885693
1,2338,459.616897
2,8531,857.707090
3,8952,353.576044
4,11064,259.190848


In [59]:
# sanity checks against sample_submission.csv format
sample_sub = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample_sub.columns), 'column mismatch vs sample_submission'
assert len(submission) == len(sample_sub), 'row count mismatch vs sample_submission'
assert submission['Property_id'].isin(sample_sub['Property_id']).all(), 'Property_id mismatch vs sample_submission'
print('rent prediction stats:')
print(submission['rent'].describe())

submission.to_csv('submission.csv', index=False)
print('saved submission.csv')

rent prediction stats:
count    3527.000000
mean      553.696495
std       311.974758
min        40.913097
25%       338.951110
50%       453.751841
75%       681.074424
max      2488.933643
Name: rent, dtype: float64
saved submission.csv
